In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

# Locate the project root
cwd = Path.cwd()

if (cwd / "data" / "raw" / "ai4i2020.csv").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "raw" / "ai4i2020.csv").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Cannot find data/raw/ai4i2020.csv. "
        "Check the project folder structure."
    )

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
MODELS_DIR = ARTIFACTS_DIR / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)
print("Reports directory:", REPORTS_DIR)
print("Models directory:", MODELS_DIR)


Project root: d:\robotic-predictive-maintenance
Dataset path: d:\robotic-predictive-maintenance\data\raw\ai4i2020.csv
Reports directory: d:\robotic-predictive-maintenance\artifacts\reports
Models directory: d:\robotic-predictive-maintenance\artifacts\models


In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

assert df.shape == (10000, 14), (
    f"Unexpected dataset shape: {df.shape}"
)

assert df["UDI"].is_unique, "UDI should be unique."
assert df["Machine failure"].isin([0, 1]).all(), (
    "Target must contain only 0 and 1."
)

print("\nTarget distribution:")
display(
    df["Machine failure"]
    .value_counts()
    .sort_index()
    .rename_axis("Machine failure")
    .to_frame("count")
)


Dataset shape: (10000, 14)
Missing values: 0
Duplicate rows: 0

Target distribution:


,count
Machine failure,
0,9661
1,339


In [3]:
TARGET_COLUMN = "Machine failure"

NUMERICAL_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

CATEGORICAL_FEATURES = [
    "Type",
]

FEATURE_COLUMNS = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

EXCLUDED_COLUMNS = [
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("Input features:")
display(pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "dtype": [X[column].dtype for column in FEATURE_COLUMNS],
}))

print("\nExcluded columns:", EXCLUDED_COLUMNS)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target positive rate:", f"{y.mean() * 100:.3f}%")

assert TARGET_COLUMN not in X.columns
assert all(column not in X.columns for column in EXCLUDED_COLUMNS)
assert X.columns.tolist() == FEATURE_COLUMNS


Input features:


,feature,dtype
0,Air temperature [K],float64
1,Process temperature [K],float64
2,Rotational speed [rpm],int64
3,Torque [Nm],float64
4,Tool wear [min],int64
5,Type,str



Excluded columns: ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
X shape: (10000, 6)
y shape: (10000,)
Target positive rate: 3.390%


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
display(
    y_train.value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .to_frame("count")
    .assign(
        percentage=lambda table: (
            table["count"] / len(y_train) * 100
        )
    )
)

print("\nTest target distribution:")
display(
    y_test.value_counts()
    .sort_index()
    .rename_axis(TARGET_COLUMN)
    .to_frame("count")
    .assign(
        percentage=lambda table: (
            table["count"] / len(y_test) * 100
        )
    )
)


Training set shape: (8000, 6)
Test set shape: (2000, 6)

Training target distribution:


,count,percentage
Machine failure,,
0,7729,96.6125
1,271,3.3875



Test target distribution:


,count,percentage
Machine failure,,
0,1932,96.6000
1,68,3.4000


In [5]:
assert X_train.index.intersection(X_test.index).empty
assert y_train.index.equals(X_train.index)
assert y_test.index.equals(X_test.index)

split_summary = pd.DataFrame({
    "observations": [
        len(y_train),
        len(y_test),
    ],
    "failures": [
        int(y_train.sum()),
        int(y_test.sum()),
    ],
    "failure_rate_pct": [
        y_train.mean() * 100,
        y_test.mean() * 100,
    ],
}, index=["Train", "Test"])

display(split_summary.round(3))

print("Train/test index overlap:", len(X_train.index.intersection(X_test.index)))
print("Original failure rate:", f"{y.mean() * 100:.3f}%")
print("Training failure rate:", f"{y_train.mean() * 100:.3f}%")
print("Test failure rate:", f"{y_test.mean() * 100:.3f}%")

assert len(X_train) + len(X_test) == len(df)
assert X_train.index.intersection(X_test.index).empty

print("\nTrain/test split validation passed.")


,observations,failures,failure_rate_pct
Train,8000,271,3.3880
Test,2000,68,3.4000


Train/test index overlap: 0
Original failure rate: 3.390%
Training failure rate: 3.388%
Test failure rate: 3.400%

Train/test split validation passed.


In [6]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

print("Numerical preprocessing pipeline created.")


Numerical preprocessing pipeline created.


In [7]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
        ),
    ]
)

print("Categorical preprocessing pipeline created.")


Categorical preprocessing pipeline created.


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERICAL_FEATURES,
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print("Combined preprocessing pipeline:")
display(preprocessor)


Combined preprocessing pipeline:


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per tran

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index,
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index,
)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

print("\nTransformed feature names:")
print(feature_names.tolist())

display(X_train_processed.head())


Processed training shape: (8000, 8)
Processed test shape: (2000, 8)

Transformed feature names:
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Type_H', 'Type_L', 'Type_M']


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Type_H,Type_L,Type_M
4058,0.9989,0.6043,-0.4606,0.7183,-0.8440,0.0000,0.0000,1.0000
1221,-1.5052,-1.1533,-0.7756,0.6385,0.3823,0.0000,0.0000,1.0000
6895,0.4981,1.0775,-1.0077,0.5586,0.4609,0.0000,0.0000,1.0000
9863,-0.5536,-0.1393,-0.7093,1.6266,-0.3724,0.0000,1.0000,0.0000
8711,-1.4551,-1.0181,1.0700,-1.1282,-0.9069,0.0000,1.0000,0.0000


In [11]:

FAILURE_MODE_COLUMNS = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

print("Failure-mode columns defined:", FAILURE_MODE_COLUMNS)


assert X_train_processed.shape[0] == len(X_train)
assert X_test_processed.shape[0] == len(X_test)

assert X_train_processed.columns.tolist() == X_test_processed.columns.tolist()

assert np.isfinite(X_train_processed.to_numpy()).all()
assert np.isfinite(X_test_processed.to_numpy()).all()

assert X_train_processed.isnull().sum().sum() == 0
assert X_test_processed.isnull().sum().sum() == 0

assert TARGET_COLUMN not in X_train_processed.columns
assert all(
    column not in X_train_processed.columns
    for column in FAILURE_MODE_COLUMNS
)

print("Processed data validation passed.")
print("All transformed values are finite.")
print("No missing values remain.")
print("Train and test feature schemas match.")
print("No target or failure-mode labels are present in the inputs.")


Failure-mode columns defined: ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
Processed data validation passed.
All transformed values are finite.
No missing values remain.
Train and test feature schemas match.
No target or failure-mode labels are present in the inputs.


In [12]:
import joblib
import json

# Save the fitted preprocessing pipeline
preprocessor_path = MODELS_DIR / "preprocessor.joblib"

joblib.dump(preprocessor, preprocessor_path)

print(f"Preprocessor saved to: {preprocessor_path}")

Preprocessor saved to: d:\robotic-predictive-maintenance\artifacts\models\preprocessor.joblib


In [13]:
split_metadata = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "target_column": TARGET_COLUMN,
    "feature_columns": FEATURE_COLUMNS,
    "numerical_features": NUMERICAL_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "excluded_columns": EXCLUDED_COLUMNS,
    "train_observations": len(X_train),
    "test_observations": len(X_test),
    "train_failures": int(y_train.sum()),
    "test_failures": int(y_test.sum()),
    "transformed_feature_names": feature_names.tolist(),
}

metadata_path = REPORTS_DIR / "preprocessing_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(split_metadata, file, indent=4)

print(f"Metadata saved to: {metadata_path}")


Metadata saved to: d:\robotic-predictive-maintenance\artifacts\reports\preprocessing_metadata.json


In [14]:
processed_dir = ARTIFACTS_DIR / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

X_train_path = processed_dir / "X_train.csv"
X_test_path = processed_dir / "X_test.csv"
y_train_path = processed_dir / "y_train.csv"
y_test_path = processed_dir / "y_test.csv"

X_train_processed.to_csv(X_train_path, index_label="original_index")
X_test_processed.to_csv(X_test_path, index_label="original_index")

y_train.rename(TARGET_COLUMN).to_csv(
    y_train_path,
    index_label="original_index",
)

y_test.rename(TARGET_COLUMN).to_csv(
    y_test_path,
    index_label="original_index",
)

print("Saved processed datasets:")
for path in [X_train_path, X_test_path, y_train_path, y_test_path]:
    print(f"  {path}")


Saved processed datasets:
  d:\robotic-predictive-maintenance\artifacts\processed\X_train.csv
  d:\robotic-predictive-maintenance\artifacts\processed\X_test.csv
  d:\robotic-predictive-maintenance\artifacts\processed\y_train.csv
  d:\robotic-predictive-maintenance\artifacts\processed\y_test.csv


In [15]:
loaded_preprocessor = joblib.load(preprocessor_path)

# Transform a small sample without fitting again
sample_processed = loaded_preprocessor.transform(X_test.iloc[:5])

assert sample_processed.shape[0] == 5
assert sample_processed.shape[1] == len(feature_names)
assert np.isfinite(sample_processed).all()

print("Saved preprocessor reloaded successfully.")
print("Sample transformed shape:", sample_processed.shape)
print("Feature count:", len(feature_names))


Saved preprocessor reloaded successfully.
Sample transformed shape: (5, 8)
Feature count: 8


In [16]:
assert len(X_train_processed) == 8000
assert len(X_test_processed) == 2000

assert int(y_train.sum() + y_test.sum()) == int(y.sum())

assert set(X_train.index).isdisjoint(set(X_test.index))

assert preprocessor_path.exists()
assert metadata_path.exists()

for path in [X_train_path, X_test_path, y_train_path, y_test_path]:
    assert path.exists(), f"Missing artifact: {path}"

print("PHASE 3 VALIDATION PASSED")
print("- Stratified train/test split created.")
print("- Preprocessing fitted on training data only.")
print("- Training and test schemas match.")
print("- No target or failure-mode leakage.")
print("- Fitted preprocessor saved and reloaded.")
print("- Processed datasets and metadata saved.")


PHASE 3 VALIDATION PASSED
- Stratified train/test split created.
- Preprocessing fitted on training data only.
- Training and test schemas match.
- No target or failure-mode leakage.
- Fitted preprocessor saved and reloaded.
- Processed datasets and metadata saved.
